In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
%%capture

!pip install transformers
!pip install datasets
!pip install evaluate

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import os

In [9]:
dataset = load_dataset("json", data_files={"train": "/content/treino.jsonl", "test": "/content/teste.jsonl"})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 500
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 100
    })
})

In [12]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

mapDict = {
    "venda": 0,
    "suporte": 1,
}

def transform_labels(label):
  label = label['completion']
  result = []
  for l in label:
    result.append(mapDict[l])
  return {"label": result}

def tokenize_function(example):
  return tokenizer(example['prompt'], padding=True, truncation=True)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
from transformers import TrainingArguments

output_dir = "./bert-client-support"

training_args = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    weight_decay = 0.01,
    logging_dir = "./logs",
    logging_steps = 100,
    eval_strategy = "steps",
    eval_steps = 200,
    save_total_limit = 2,
    save_steps = 200,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    report_to = "none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
from transformers import Trainer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 3)

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'offline'

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions = predictions, references = labels)

In [18]:
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['test'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metric
)

In [19]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.0553947017779426, metrics={'train_runtime': 528.4619, 'train_samples_per_second': 2.838, 'train_steps_per_second': 0.358, 'total_flos': 20041842366000.0, 'train_loss': 0.0553947017779426, 'epoch': 3.0})

In [20]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.0005487569142132998,
 'eval_accuracy': 1.0,
 'eval_runtime': 8.2522,
 'eval_samples_per_second': 12.118,
 'eval_steps_per_second': 1.575,
 'epoch': 3.0}

In [21]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
trainer.push_to_hub("guilchaves/bert-supportassistant")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...support/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...support/training_args.bin:   1%|1         |  67.0B / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/guilchaves/bert-client-support/commit/f9a0872d7032d9042e14ba00c1fb053f3616d8c3', commit_message='guilchaves/bert-supportassistant', commit_description='', oid='f9a0872d7032d9042e14ba00c1fb053f3616d8c3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/guilchaves/bert-client-support', endpoint='https://huggingface.co', repo_type='model', repo_id='guilchaves/bert-client-support'), pr_revision=None, pr_num=None)

In [25]:
from transformers import pipeline

pipe = pipeline("text-classification", model="guilchaves/bert-client-support")

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [31]:
pipe("Quero comprar um celular")

[{'label': 'LABEL_0', 'score': 0.9994363188743591}]

In [32]:
pipe("Preciso devolver um produto.")

[{'label': 'LABEL_1', 'score': 0.957802951335907}]

In [33]:
pipe("Preciso de ajuda com meu cadastro")

[{'label': 'LABEL_1', 'score': 0.999426007270813}]

In [34]:
pipe("Qual a melhor televisão?")

[{'label': 'LABEL_0', 'score': 0.9929324388504028}]